In [1]:
### IMPORTS
import pandas as pd
import numpy as np
import os 
from backtester.pnl_numba import pnl
from backtester.metrics import compute_metrics
from utils.io import load_partitioned_parquet
from utils.paths import make_data_path
print("DONE")

###CONFIG PLEASE
MAKER_FEE = 0.000200
TAKER_FEE = 0.000550
FUNDING_INTERVAL = 480 # 8 hours in minutes
SYMBOL = 'BTCUSDT'
INTERVAL = 60
START = '01/01/2026'
END = None

DONE


In [2]:
### THis needs to be a function too
### LOAD DATA

cols = ['mark_close'] # Whatever I want for the strategy 
path_ohlcv = make_data_path('ohlcv', SYMBOL, INTERVAL)
df_ohlcv = load_partitioned_parquet(path_ohlcv, start=START, end=END, columns=cols)

if df_ohlcv.empty:
    raise ValueError("df_ohlcv loaded empty. Check path or date filters.")

path_funding = make_data_path('funding', SYMBOL)
df_funding = load_partitioned_parquet(path_funding,start=START, end=END)

if df_funding.empty:
    raise ValueError("df_funding loaded empty. Check path or date filters.")

df_merge = df_ohlcv.merge(df_funding, how='left', on='timestamp')
df_merge['fundingRate'] = df_merge['fundingRate'].fillna(0)
df_merge = df_merge.sort_index()

mark_close = df_merge["mark_close"].astype("float64").to_numpy()
funding  = df_merge["fundingRate"].astype("float64").to_numpy()


In [ ]:
pos = np.ones(len(mark_close)) # always long lol
print(len(pos))

In [5]:
pos = np.zeros_like(mark_close)
for i in range(len(mark_close)):
    if i % 2 == 0:
        pos[i] = 1

In [4]:
pos = np.random.uniform(-1, 1, size=(len(mark_close)))

In [ ]:
pnl_df = pnl(df_merge, pos)
pnl_df['returns_normalized (%)']*=100
print(pnl_df.head(10))

KeyError: 'returns_normalized'